# Predicting ad positioning using Q-Learning
Placement of ads on website is the primary problem for companies that operate on ad revenue. The position where the ad is placed plays pivotal role on whether or not the ad will be clicked. Here we have the following choices:
1. Place them randomly, or
2. Place the ad on the same position

The problem with placing the ad on the same position is the user, after a certain time, will start ignoring the space since he's used to seeing ad at the place, he will end up ignoring that particular position hereafter. Hence, this will reduce the number of clicks on ads. The problem with the former option, placing them randomly, is it wouldn't take optimal positions into consideration. For instance, text beside images are viewed higher number of times than those text which are placed at a distance. It is infeasible to go through every website and repeat the procedure.

Solution: Reinforcement Learning

Using Reinforcement Learning we can approximate the human behavior.

### Why Reinforcement Learning?
We cannot use traditional Machine Learning here, since it requires:
1. Huge data
2. Features
3. Tuning of many hyperparameters

And we neither have huge data, nor features. The only data we have is the position of the banner/ad and whether or not it was clicked. We will use this dataset from Kaggle: https://www.kaggle.com/akram24/ads-ctr-optimisation. We will solve this problem using both model-based (Policy iteration) and model-free methods(Q-Learning & Monte-Carlo). We'll simplify some assumptions for model-based technique.

### Notebook Layout
1. MDP environment - Understanding the dataset
2. Random Policy - placing the ads randomly on different webpages
3. Max policy - placing the ad where it is clicked maximum number of times
4. Model-based Method - Policy Iteration
5. Model-free Method - Q-learning & Monte Carlo

## Importing Libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

In [2]:
df = pd.read_csv("Ads_CTR_Optimisation.csv")

In [3]:
df

,Ad 1,Ad 2,Ad 3,Ad 4,Ad 5,Ad 6,Ad 7,Ad 8,Ad 9,Ad 10
0,1,0,0,0,1,0,0,0,1,0
1,0,0,0,0,0,0,0,0,1,0
2,0,0,0,0,0,0,0,0,0,0
3,0,1,0,0,0,0,0,1,0,0
4,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...
9995,0,0,1,0,0,0,0,1,0,0
9996,0,0,0,0,0,0,0,0,0,0
9997,0,0,0,0,0,0,0,0,0,0
9998,1,0,0,0,0,0,0,1,0,0


## Dataset

Our environment will be the dataset. It contains 10 ads position per row having values either 1, when the ad is clicked, or 0 when it is not. Every row can be considered as a state in the space, considering it kind of a navigation across multiple pages (on website, for instance) Lets load the dataset and visualize the first few rows.

state = webpage

action = placing the ad at any of the 10 positions on a webpage

reward = +1 if the ad was clicked at the position; else 0

Transition Probability: next webpage that user will end up in is random; therefore, it is 1/(total_webpages -1)

In [4]:
env = df

In [5]:
env.head()

,Ad 1,Ad 2,Ad 3,Ad 4,Ad 5,Ad 6,Ad 7,Ad 8,Ad 9,Ad 10
0,1,0,0,0,1,0,0,0,1,0
1,0,0,0,0,0,0,0,0,1,0
2,0,0,0,0,0,0,0,0,0,0
3,0,1,0,0,0,0,0,1,0,0
4,0,0,0,0,0,0,0,0,0,0


In [6]:
env["Ad 1"].describe()

count    10000.000000
mean         0.170300
std          0.375915
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          1.000000
Name: Ad 1, dtype: float64

In [7]:
(env.iloc[0] == 1).any()

np.True_

In [8]:
urls_clicked = dict()
for i in range(len(env)):
    urls_clicked[i] = 1 if (env.iloc[i] == 1).any() else 0

In [11]:
sum(list(urls_clicked.values())) / len(env)

0.7424

In [12]:
urls_clicked.items()

dict_items([(0, 1), (1, 1), (2, 0), (3, 1), (4, 0), (5, 1), (6, 1), (7, 1), (8, 0), (9, 1), (10, 0), (11, 0), (12, 1), (13, 1), (14, 1), (15, 1), (16, 0), (17, 0), (18, 1), (19, 1), (20, 1), (21, 1), (22, 0), (23, 1), (24, 1), (25, 0), (26, 1), (27, 1), (28, 0), (29, 1), (30, 1), (31, 0), (32, 1), (33, 1), (34, 1), (35, 0), (36, 0), (37, 0), (38, 1), (39, 1), (40, 1), (41, 1), (42, 1), (43, 1), (44, 1), (45, 1), (46, 0), (47, 0), (48, 1), (49, 1), (50, 1), (51, 0), (52, 1), (53, 1), (54, 1), (55, 1), (56, 1), (57, 1), (58, 0), (59, 1), (60, 1), (61, 1), (62, 1), (63, 1), (64, 1), (65, 1), (66, 1), (67, 1), (68, 0), (69, 0), (70, 0), (71, 1), (72, 1), (73, 0), (74, 1), (75, 0), (76, 0), (77, 1), (78, 1), (79, 0), (80, 1), (81, 1), (82, 1), (83, 1), (84, 1), (85, 1), (86, 0), (87, 1), (88, 1), (89, 1), (90, 1), (91, 1), (92, 0), (93, 0), (94, 1), (95, 0), (96, 1), (97, 1), (98, 1), (99, 1), (100, 1), (101, 0), (102, 1), (103, 1), (104, 1), (105, 1), (106, 0), (107, 1), (108, 1), (109, 1)